# 04 — LLM models (GPT-2, ParsBERT, System One, Laya)

The four pretrained-model pipelines behind the serving stack:

- **GPT-2-fa** — HooshvareLab/gpt2-fa fully fine-tuned on GPU (all 12 blocks, 128 tokens of context), pooled on the last real token (causal).
- **ParsBERT** — HooshvareLab/bert-fa-base-uncased fine-tuned end to end on GPU, pooled on [CLS] (bidirectional).
- **System One** — the fine-tuned ParsBERT encoder feeding three typed decision heads
  (choice / score / noul), trained by distilling soft targets from the GPT-2 classifier.
- **Laya** — the open-weights mmBERT System One agent, fine-tuned with the shipped RLCD loop.

By default every section **loads the deployed checkpoint and scores it through the exact serving
path** (`aitextd.loaders` + `aitextd.predictors`), so the notebook reports what the API serves.
Set the `FORCE_*` flags to re-run any training stage from scratch.


In [ ]:
import gc
import json
import os

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from aitextd import loaders, paths, predictors
from aitextd.constants import SYSTEM_ONE_DETECTION_QUESTIONS
from aitextd.features import compute_style_features
from aitextd.system_one import typed_answers
from aitextd.text import normalize_for_pretrained

DATA = paths.PROJECT_ROOT / "data"
RANDOM_SEED = 42
SEQ_LEN = 64
BATCH_SIZE = 16
TRAIN_SUBSET, TEST_SUBSET = 6000, 1500  # CPU budget, same as the original trainers
EVAL_SAMPLES = 300  # deterministic stratified subsample for the report table
FORCE_TRAIN_GPT2 = False
FORCE_TRAIN_PARSBERT = False
FORCE_BUILD_S1_DATA = False
FORCE_TRAIN_S1 = False
FORCE_TRAIN_LAYA = False

df = pd.read_csv(paths.PROCESSED_CSV)
y = df["aiGenerated"].to_numpy()
texts = df["text"].to_numpy()
split = joblib.load(DATA / "split_indices.joblib")
train_idx, test_idx = split["train_idx"], split["test_idx"]
vectorizer = joblib.load(paths.TFIDF_VECTORIZER_JOBLIB)


def subsample(indices, labels, n, seed):
    """Deterministic stratified subsample."""
    if n >= len(indices):
        return indices
    sub, _ = train_test_split(indices, train_size=n, random_state=seed, stratify=labels[indices])
    return sub


def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i : i + size]


eval_idx = np.sort(subsample(test_idx, y, EVAL_SAMPLES, RANDOM_SEED))
y_eval = y[eval_idx]


def metrics_from_probs(probs, labels):
    """Accuracy, AI-class P/R/F1, ROC-AUC and 10-bin ECE from AI probabilities."""
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

    probs, labels = np.asarray(probs), np.asarray(labels)
    preds = (probs >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1], zero_division=0)
    bins = np.minimum((probs * 10).astype(int), 9)
    ece = 0.0
    for b in range(10):
        mask = bins == b
        if mask.any():
            ece += mask.mean() * abs(labels[mask].mean() - probs[mask].mean())
    return {
        "accuracy": round(float(accuracy_score(labels, preds)), 4),
        "precision": round(float(p[1]), 4),
        "recall": round(float(r[1]), 4),
        "f1Score": round(float(f1[1]), 4),
        "rocAuc": round(float(roc_auc_score(labels, probs)), 4),
        "ece": round(float(ece), 4),
    }


def serving_probs(model_id, indices):
    """AI probabilities via the exact serving path (same loader/predictor as the API)."""
    registry = json.load(open(paths.MODELS_JSON, encoding="utf-8"))
    md = next(m for m in registry if m["id"] == model_id)
    bundle = loaders.LOADERS[md["runtime"]](md)
    predict = predictors.PREDICTORS[md["runtime"]]
    probs = []
    for i in indices:
        text = str(texts[i])
        cleaned = normalize_for_pretrained(text)
        style = compute_style_features(text, cleaned, vectorizer)
        result = predict(bundle, text, cleaned, style)
        probs.append(result["aiProbability"] if isinstance(result, dict) else float(result))
    return np.asarray(probs), bundle


def register_entry(entry):
    """Merge one entry into models.json (existing ensemble weights survive)."""
    existing = json.load(open(paths.MODELS_JSON, encoding="utf-8"))
    prior = next((m for m in existing if m["id"] == entry["id"]), {})
    entry.setdefault("ensembleWeight", prior.get("ensembleWeight", 1.0))
    existing = [m for m in existing if m["id"] != entry["id"]] + [entry]
    with open(paths.MODELS_JSON, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)
    print(f"models.json entry '{entry['id']}' updated")


results = {}
torch.set_num_threads(12)


## GPT-2 classifier

The locally pickled Persian GPT-2 (`gpt2-fa.mdl`). The deployed checkpoint was fully
fine-tuned on GPU (all 12 blocks, 128 tokens of context, mixed precision, validation-based
early stopping); the FORCE path below is a CPU-budget reproduction of it — last 4 blocks
(plus `ln_f` and the head) unfrozen, one epoch over a stratified 6000/1500 subsample.


In [ ]:
MODEL_FILE = "gpt2ClassifierModel.pt"

if FORCE_TRAIN_GPT2:
    import pickle

    import torch.nn as nn

    from aitextd.architectures import GPT2Classifier

    with open(paths.GPT2_LM_MDL, "rb") as f:
        lm = pickle.load(f)
    with open(paths.GPT2_TOKENIZER_MDL, "rb") as f:
        gpt2_tok = pickle.load(f)
    gpt2_tok.pad_token = "<pad>"
    backbone = lm.transformer
    hidden_size = backbone.config.n_embd
    for p in backbone.parameters():
        p.requires_grad = False
    for block in backbone.h[-4:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in backbone.ln_f.parameters():
        p.requires_grad = True

    train_sub = subsample(train_idx, y, TRAIN_SUBSET, RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    model = GPT2Classifier(backbone, hidden_size)
    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in backbone.parameters() if p.requires_grad], "lr": 2e-5},
            {"params": model.head.parameters(), "lr": 1e-3},
        ]
    )
    criterion = nn.BCEWithLogitsLoss()
    char_budget = SEQ_LEN * 8  # the pickled tokenizer is slow: pre-truncate per character

    def encode(idx_batch):
        enc = gpt2_tok(
            [normalize_for_pretrained(str(texts[i]))[:char_budget] for i in idx_batch],
            return_tensors="pt", truncation=True, max_length=SEQ_LEN, padding="max_length",
        )
        return enc["input_ids"], enc["attention_mask"], torch.tensor(y[idx_batch], dtype=torch.float32)

    model.train()
    order = train_sub.copy()
    np.random.default_rng(RANDOM_SEED + 1).shuffle(order)
    for step, idx_batch in enumerate(batches(order, BATCH_SIZE), start=1):
        input_ids, attention_mask, labels = encode(idx_batch)
        optimizer.zero_grad()
        loss = criterion(model(input_ids, attention_mask), labels)
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(f"step {step} loss={loss.item():.4f}", flush=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": {
                "seq_len": SEQ_LEN,
                "hidden_size": hidden_size,
                "n_layer": backbone.config.n_layer,
                "vocab_size": backbone.config.vocab_size,
                "n_head": backbone.config.n_head,
            },
        },
        paths.model_path(MODEL_FILE),
    )
    del model, backbone, lm
    gc.collect()
    print("checkpoint saved")
else:
    print("loading deployed gpt2ClassifierModel.pt")

probs, _ = serving_probs("gpt2_transformer", eval_idx)
results["gpt2_transformer"] = metrics_from_probs(probs, y_eval)
del probs
gc.collect()
print(results["gpt2_transformer"])


## ParsBERT classifier

Bidirectional Persian BERT, pooled on [CLS]. The deployed checkpoint was fine-tuned end to
end on a Colab GPU over the full dataset; the FORCE path below retrains under the same CPU
budget as the GPT-2 trainer so the two rows stay directly comparable. The saved checkpoint
stores the backbone config, which is how the server rebuilds the encoder offline.


In [ ]:
MODEL_FILE = "parsbertClassifierModel.pt"
BACKBONE_NAME = "HooshvareLab/bert-fa-base-uncased"

if FORCE_TRAIN_PARSBERT:
    import torch.nn as nn
    from transformers import AutoModel, AutoTokenizer

    from aitextd.architectures import ParsBERTClassifier

    tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)
    backbone = AutoModel.from_pretrained(BACKBONE_NAME)
    hidden_size = backbone.config.hidden_size
    for p in backbone.parameters():
        p.requires_grad = False
    for layer in backbone.encoder.layer[-4:]:
        for p in layer.parameters():
            p.requires_grad = True
    for p in backbone.pooler.parameters():
        p.requires_grad = True

    train_sub = subsample(train_idx, y, TRAIN_SUBSET, RANDOM_SEED)
    test_sub = subsample(test_idx, y, TEST_SUBSET, RANDOM_SEED)
    needed = np.concatenate([train_sub, test_sub])
    normed = {int(i): normalize_for_pretrained(str(texts[i])) for i in needed}
    torch.manual_seed(RANDOM_SEED)
    model = ParsBERTClassifier(backbone, hidden_size)
    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in backbone.parameters() if p.requires_grad], "lr": 2e-5},
            {"params": model.head.parameters(), "lr": 1e-3},
        ],
        weight_decay=0.01,
    )
    criterion = nn.BCEWithLogitsLoss()

    def encode(idx_batch):
        enc = tokenizer(
            [normed[int(i)] for i in idx_batch],
            return_tensors="pt", truncation=True, max_length=SEQ_LEN, padding="max_length",
        )
        return enc["input_ids"], enc["attention_mask"], torch.tensor(y[idx_batch], dtype=torch.float32)

    model.train()
    order = train_sub.copy()
    np.random.default_rng(RANDOM_SEED + 1).shuffle(order)
    for step, idx_batch in enumerate(batches(order, BATCH_SIZE), start=1):
        input_ids, attention_mask, labels = encode(idx_batch)
        optimizer.zero_grad()
        loss = criterion(model(input_ids, attention_mask), labels)
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(f"step {step} loss={loss.item():.4f}", flush=True)
    backbone_config = backbone.config.to_dict()
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": {
                "seq_len": SEQ_LEN,
                "hidden_size": hidden_size,
                "n_layer": backbone.config.num_hidden_layers,
                "backbone_config": backbone_config,  # the server rebuilds the encoder from this
            },
        },
        paths.model_path(MODEL_FILE),
    )
    tokenizer.save_pretrained(paths.PARSBERT_TOKENIZER_DIR)
    del model, backbone
    gc.collect()
    print("checkpoint saved")
else:
    print("loading deployed parsbertClassifierModel.pt")

probs, _ = serving_probs("parsbert", eval_idx)
results["parsbert"] = metrics_from_probs(probs, y_eval)
del probs
gc.collect()
print(results["parsbert"])


## System One: distillation data

A System One model answers typed questions instead of generating text. Each row becomes a
JSONL record with three typed questions — `source` (choice), `ai_level` (score over 5
ordinal levels), `needs_review` (noul) — whose **soft targets are distilled from the deployed
GPT-2 classifier** (the ensemble's heaviest weight, tied with ParsBERT) through the exact serving path.
Loaded from `data/system_one_data/*.jsonl` unless `FORCE_BUILD_S1_DATA` is set.


In [ ]:
S1_DATA = DATA / "system_one_data"

if FORCE_BUILD_S1_DATA:
    from aitextd.constants import (
        SYSTEM_ONE_CHOICE_CANDIDATES,
        SYSTEM_ONE_CHOICE_CANDIDATES_FA,
        SYSTEM_ONE_SCORE_LEVELS,
        SYSTEM_ONE_SCORE_LEVELS_FA,
    )
    from aitextd.loaders import get_gpt2_tokenizer, load_gpt2_backbone
    from aitextd.architectures import GPT2Classifier

    QUESTIONS = {
        "source": {"type": "choice", "candidates": SYSTEM_ONE_CHOICE_CANDIDATES, "candidatesFa": SYSTEM_ONE_CHOICE_CANDIDATES_FA},
        "ai_level": {"type": "score", "levels": SYSTEM_ONE_SCORE_LEVELS, "levelsFa": SYSTEM_ONE_SCORE_LEVELS_FA},
        "needs_review": {"type": "noul"},
    }
    LABEL_WEIGHT, TEACHER_WEIGHT = 0.8, 0.2
    CHOICE_CLIP, SCORE_SIGMA, NOUL_BAND = (0.02, 0.98), 0.9, (0.15, 0.85)

    teacher_ckpt = torch.load(paths.model_path("gpt2ClassifierModel.pt"), map_location="cpu", weights_only=False)
    teacher = GPT2Classifier(load_gpt2_backbone(), hidden_size=teacher_ckpt["config"]["hidden_size"])
    teacher.load_state_dict(teacher_ckpt["state_dict"])
    teacher.eval()
    tok = get_gpt2_tokenizer()
    char_budget = teacher_ckpt["config"]["seq_len"] * 8

    def teacher_probs(raw_texts):
        out = []
        with torch.no_grad():
            for batch in batches(raw_texts, 32):
                enc = tok(
                    [normalize_for_pretrained(str(t))[:char_budget] for t in batch],
                    return_tensors="pt", truncation=True,
                    max_length=teacher_ckpt["config"]["seq_len"], padding="max_length",
                )
                out.append(torch.sigmoid(teacher(enc["input_ids"], enc["attention_mask"])).numpy())
        return np.concatenate(out)

    def build_records(indices, split_name):
        p_teacher = teacher_probs(texts[indices])
        records = []
        for k, i in enumerate(indices):
            t_ai = min(max(LABEL_WEIGHT * float(y[i]) + TEACHER_WEIGHT * float(p_teacher[k]), *CHOICE_CLIP), 1.0)
            center = (len(SYSTEM_ONE_SCORE_LEVELS) - 1) * t_ai
            levels = np.arange(len(SYSTEM_ONE_SCORE_LEVELS), dtype=float)
            score_t = np.exp(-((levels - center) ** 2) / (2 * SCORE_SIGMA**2))
            score_t = score_t / score_t.sum()
            records.append({
                "id": f"{split_name}-{i:06d}",
                "split": split_name,
                "label": int(y[i]),
                "state": {"text": str(texts[i])},
                "questions": QUESTIONS,
                "teacher": "gpt2_transformer",
                "targets": {
                    "source": {"human_written": round(1 - t_ai, 4), "ai_generated": round(t_ai, 4)},
                    "ai_level": [round(float(p), 4) for p in score_t],
                    "needs_review": 1.0 if NOUL_BAND[0] <= p_teacher[k] <= NOUL_BAND[1] else 0.0,
                },
                "teacherAiProbability": round(float(p_teacher[k]), 4),
            })
        return records

    S1_DATA.mkdir(parents=True, exist_ok=True)
    train_sub = subsample(train_idx, y, TRAIN_SUBSET, RANDOM_SEED)
    test_sub = subsample(test_idx, y, TEST_SUBSET, RANDOM_SEED)
    for name, sub in (("train", train_sub), ("test", test_sub)):
        with open(S1_DATA / f"system_one_{name}.jsonl", "w", encoding="utf-8") as f:
            for record in build_records(sub, name):
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"{name}: wrote {len(sub)} records")
    del teacher
    gc.collect()
else:
    n_train = sum(1 for _ in open(S1_DATA / "system_one_train.jsonl", encoding="utf-8"))
    n_test = sum(1 for _ in open(S1_DATA / "system_one_test.jsonl", encoding="utf-8"))
    print(f"using existing JSONL targets: {n_train} train / {n_test} test records")


## System One: decision-head training

The fine-tuned ParsBERT encoder is reused as the backbone; its [CLS] representation feeds
three heads trained with soft cross-entropy on the distilled distributions (+ BCE on the
noul head). Three epochs over the same 6000/1500 subsample, last 4 encoder layers unfrozen.


In [ ]:
if FORCE_TRAIN_S1:
    import torch.nn as nn
    from itertools import pairwise
    from transformers import AutoModel, BertConfig

    from aitextd.architectures import ParsBERTClassifier, SystemOneDecisionModel
    from aitextd.constants import SYSTEM_ONE_CHOICE_CANDIDATES, SYSTEM_ONE_SCORE_LEVELS
    from aitextd.loaders import get_parsbert_tokenizer

    def load_jsonl(path):
        return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

    train_records = load_jsonl(S1_DATA / "system_one_train.jsonl")
    test_records = load_jsonl(S1_DATA / "system_one_test.jsonl")
    choice_t = np.array([[r["targets"]["source"][c] for c in SYSTEM_ONE_CHOICE_CANDIDATES] for r in train_records], dtype=np.float32)
    score_t = np.array([r["targets"]["ai_level"] for r in train_records], dtype=np.float32)
    noul_t = np.array([r["targets"]["needs_review"] for r in train_records], dtype=np.float32)
    y_test_s1 = np.array([r["label"] for r in test_records])
    normed = [normalize_for_pretrained(r["state"]["text"]) for r in train_records]
    normed_test = [normalize_for_pretrained(r["state"]["text"]) for r in test_records]

    base_ckpt = torch.load(paths.model_path("parsbertClassifierModel.pt"), map_location="cpu", weights_only=False)
    backbone = AutoModel.from_config(BertConfig.from_dict(base_ckpt["config"]["backbone_config"]))
    classifier = ParsBERTClassifier(backbone, base_ckpt["config"]["hidden_size"])
    classifier.load_state_dict(base_ckpt["state_dict"])
    backbone = classifier.backbone  # fine-tuned classification knowledge stays intact
    hidden_size = backbone.config.hidden_size
    for p in backbone.parameters():
        p.requires_grad = False
    for layer in backbone.encoder.layer[-4:]:
        for p in layer.parameters():
            p.requires_grad = True

    torch.manual_seed(RANDOM_SEED)
    model = SystemOneDecisionModel(
        backbone, hidden_size,
        n_choice=len(SYSTEM_ONE_CHOICE_CANDIDATES),
        n_score_levels=len(SYSTEM_ONE_SCORE_LEVELS),
        pool_on_cls=True,
    )
    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in backbone.parameters() if p.requires_grad], "lr": 2e-5},
            {"params": model.choice_head.parameters(), "lr": 1e-3},
            {"params": model.score_head.parameters(), "lr": 1e-3},
            {"params": model.noul_head.parameters(), "lr": 1e-3},
        ],
        weight_decay=0.01,
    )

    def soft_ce(logits, target_probs):
        return -(target_probs * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

    def encode(batch_texts):
        return get_parsbert_tokenizer()(
            batch_texts, return_tensors="pt", truncation=True, max_length=SEQ_LEN, padding="max_length"
        )

    model.train()
    for epoch in (1, 2, 3):
        order = np.arange(len(train_records))
        np.random.default_rng(RANDOM_SEED + epoch).shuffle(order)
        for step, idx_batch in enumerate(batches(order, BATCH_SIZE), start=1):
            enc = encode([normed[i] for i in idx_batch])
            choice_logits, score_logits, noul_logit = model(enc["input_ids"], enc["attention_mask"])
            loss = (
                soft_ce(choice_logits, torch.from_numpy(choice_t[idx_batch]))
                + 0.5 * soft_ce(score_logits, torch.from_numpy(score_t[idx_batch]))
                + 0.5 * nn.functional.binary_cross_entropy_with_logits(noul_logit, torch.from_numpy(noul_t[idx_batch]))
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if step % 50 == 0:
                print(f"epoch {epoch} step {step} loss={loss.item():.4f}", flush=True)

    model.eval()
    choice_probs_all = []
    with torch.no_grad():
        for idx_batch in batches(np.arange(len(test_records)), 32):
            enc = encode([normed_test[i] for i in idx_batch])
            choice_logits, _, _ = model(enc["input_ids"], enc["attention_mask"])
            choice_probs_all.append(torch.softmax(choice_logits, dim=-1).numpy())
    choice_probs = np.concatenate(choice_probs_all)
    metrics = metrics_from_probs(choice_probs[:, 1], y_test_s1)
    backbone_config = backbone.config.to_dict()
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": {
                "seq_len": SEQ_LEN,
                "hidden_size": hidden_size,
                "n_choice": len(SYSTEM_ONE_CHOICE_CANDIDATES),
                "n_score_levels": len(SYSTEM_ONE_SCORE_LEVELS),
                "choice_candidates": SYSTEM_ONE_CHOICE_CANDIDATES,
                "score_levels": SYSTEM_ONE_SCORE_LEVELS,
                "thresholds": {"proceed": 0.8, "needsReview": 0.6},
                "backbone_config": backbone_config,
            },
            "metrics": metrics,
        },
        paths.model_path("systemOneModel.pt"),
    )
    del model, backbone, classifier
    gc.collect()
    print("checkpoint saved:", metrics)
else:
    print("loading deployed systemOneModel.pt")

probs, bundle_s1 = serving_probs("system_one", eval_idx)
results["system_one"] = metrics_from_probs(probs, y_eval)
print(results["system_one"])

SAMPLES = [
    "دیروز رفتم بازار تا برای عید نوروز خرید کنم. شلوغ بود و صف طولی بسته بودیم تا نان بگیریم.",
    "هوش مصنوعی یکی از مهم‌ترین فناوری‌های قرن بیست‌ویکم است. این فناوری تأثیر عمیقی بر اقتصاد گذاشته است.",
]
for text in SAMPLES:
    cleaned = normalize_for_pretrained(text)
    style = compute_style_features(text, cleaned, vectorizer)
    answers = typed_answers(bundle_s1, predictors.predict_system_one(bundle_s1, text, cleaned, style))
    print(f"choice={answers.choice.answer} score={answers.score.value:.2f} review={answers.noul.answer}")
del probs, bundle_s1
gc.collect()


## Laya

The open-weights **Laya** agent (mmBERT backbone) fine-tuned on the same System One JSONL
targets with the shipped **RLCD loop**: strictly-proper-scoring reward on noisy answer
distributions + soft cross-entropy, then per-question-type temperature calibration fitted on
held-out records. Head-only CPU mode trains the decision heads with the encoder frozen and
detached; `FORCE_TRAIN_LAYA` reproduces the deployed `models/laya_persian` checkpoint.


In [ ]:
if FORCE_TRAIN_LAYA:
    import math
    import random
    import time

    from safetensors.torch import load_file, save_file
    from transformers import AutoTokenizer
    from laya.agent import _fix_tokenizer_config
    from laya.common import (
        QTYPES, TEMP_MAX, TEMP_MIN, build_model, build_sequence,
        proper_reward, render_options, state_room,
    )

    QUESTION_TARGETS = {
        "source": ("source", "choice"),
        "machine_likeness": ("ai_level", "score"),
        "needs_review": ("needs_review", "noul"),
    }
    RECORDS, CALIB_RECORDS, TEST_RECORDS = 1200, 200, 400
    EPOCHS, MICRO_BATCH, GRAD_ACCUM = 2, 4, 8
    LR_HEAD, MAX_LEN, LAYA_SEED = 1e-4, 512, 20261005
    base_dir = paths.LAYA_BASE_DIR
    _fix_tokenizer_config(str(base_dir))

    registry = json.load(open(paths.MODELS_JSON, encoding="utf-8"))
    laya_cfg = json.load(open(base_dir / "rl_agent_config.json"))
    laya_cfg["max_len"] = MAX_LEN
    laya_cfg.setdefault("head_max_len", 256)
    tokenizer_l = AutoTokenizer.from_pretrained(base_dir / "tokenizer")

    def internal_question(qid):
        q = SYSTEM_ONE_DETECTION_QUESTIONS[qid]
        crit = q.get("criteria")
        if q["type"] == "choice" and isinstance(crit, list):
            crit = {c: None for c in crit}
        return {"t": q["type"], "ins": q["instructions"], "crit": crit}

    def target_distribution(record, qid):
        key, qtype = QUESTION_TARGETS[qid]
        value = record["targets"][key]
        if qtype == "choice":
            row = [float(value[c]) for c in ("human_written", "ai_generated")]
        elif qtype == "score":
            row = [float(p) for p in value]
        else:
            row = [1.0 - float(value), float(value)]  # noul renders [false, true]
        total = sum(row)
        return [x / total for x in row] if total > 0 else [1.0 / len(row)] * len(row)

    def build_items(records, qids):
        internal = {qid: internal_question(qid) for qid in qids}
        items = []
        for record in records:
            text = str(record["state"]["text"])
            for qid in qids:
                q = internal[qid]
                room = state_room(tokenizer_l, q, MAX_LEN, laya_cfg["head_max_len"])
                sequence, markers = build_sequence(
                    tokenizer_l, {"text": text[: max(128, room * 4)]}, q, MAX_LEN, laya_cfg["head_max_len"]
                )
                if len(markers) == len(render_options(q)):
                    items.append({
                        "ids": sequence, "markers": markers, "qtype": QTYPES[q["t"]],
                        "target": target_distribution(record, qid), "label": int(record["label"]),
                    })
        return items

    def stratified(records, n, seed):
        if n >= len(records):
            return records
        idx, _ = train_test_split(range(len(records)), train_size=n, random_state=seed, stratify=[r["label"] for r in records])
        return [records[i] for i in sorted(idx)]

    def load_jsonl(path):
        return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

    train_records = stratified(load_jsonl(S1_DATA / "system_one_train.jsonl"), RECORDS, RANDOM_SEED)
    calib_records = stratified(load_jsonl(S1_DATA / "system_one_train.jsonl"), CALIB_RECORDS, RANDOM_SEED + 1)
    train_records = [r for r in train_records if r["id"] not in {c["id"] for c in calib_records}]
    test_records = stratified(load_jsonl(S1_DATA / "system_one_test.jsonl"), TEST_RECORDS, RANDOM_SEED)
    qids = list(QUESTION_TARGETS)
    items = {name: build_items(recs, qids) for name, recs in (("train", train_records), ("calib", calib_records), ("test", test_records))}
    print({name: len(v) for name, v in items.items()})

    model = build_model(laya_cfg, encoder_dir=base_dir / "encoder")
    model.load_state_dict(load_file(str(base_dir / "model.safetensors")), strict=True)
    model.float()
    for p in model.encoder.parameters():
        p.requires_grad = False
    model.to("cpu").train()

    def collate(chunk):
        size, seq_len = len(chunk), max(len(it["ids"]) for it in chunk)
        kmax = max(len(it["markers"]) for it in chunk)
        ids = torch.full((size, seq_len), tokenizer_l.pad_token_id, dtype=torch.long)
        attention = torch.zeros((size, seq_len), dtype=torch.long)
        positions = torch.zeros((size, kmax), dtype=torch.long)
        mask = torch.zeros((size, kmax), dtype=torch.bool)
        target = torch.zeros((size, kmax), dtype=torch.float32)
        for i, it in enumerate(chunk):
            ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
            attention[i, : len(it["ids"])] = 1
            positions[i, : len(it["markers"])] = torch.tensor(it["markers"])
            mask[i, : len(it["markers"])] = True
            target[i, : len(it["target"])] = torch.tensor(it["target"])
        return ids, attention, positions, mask, target, torch.tensor([it["qtype"] for it in chunk])

    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW(head_params, lr=LR_HEAD, weight_decay=0.01)
    train_items = items["train"]
    random.Random(LAYA_SEED).shuffle(train_items)
    for epoch in range(EPOCHS):
        random.Random(RANDOM_SEED + epoch).shuffle(train_items)
        optimizer.zero_grad(set_to_none=True)
        for start in range(0, len(train_items), MICRO_BATCH):
            chunk = train_items[start : start + MICRO_BATCH]
            ids, attention, positions, mask, target, qtype = collate(chunk)
            logits, activation = model(ids, attention, positions, mask, qtype, detach_encoder=True)
            logits = logits.float()
            k = mask.sum(-1, keepdim=True).float()
            sigma = 0.4 + (0.1 - 0.4) * epoch / max(1, EPOCHS - 1)
            eps = torch.randn((4, *logits.shape)) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            noisy = logits.detach().unsqueeze(0) + eps
            probs = torch.softmax(noisy.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                reward = proper_reward(probs, target.unsqueeze(0), qtype, mask, w_sph=0.75, w_rps=1.0)
                advantage = (reward - reward.mean(0, keepdim=True)) / (reward.std() + 1e-6)
            logp = -(((noisy - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma**2)
            loss = (
                -(advantage * logp).mean()
                - (target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
                + 0.0 * activation.sum()
            ) / GRAD_ACCUM
            loss.backward()
            if (start // MICRO_BATCH + 1) % GRAD_ACCUM == 0 or start + MICRO_BATCH >= len(train_items):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

    model.eval()
    samples = [[] for _ in range(3)]
    with torch.no_grad():
        for start in range(0, len(items["calib"]), MICRO_BATCH):
            ids, attention, positions, mask, _, qtype = collate(items["calib"][start : start + MICRO_BATCH])
            logits, _ = model(ids, attention, positions, mask, qtype, detach_encoder=True)
            for i, it in enumerate(items["calib"][start : start + MICRO_BATCH]):
                samples[it["qtype"]].append((logits[i, : len(it["markers"])].float(), it["target"]))

    def fit_temperature(group):
        if len(group) < 10:
            return 1.0
        kmax = max(len(l) for l, _ in group)
        logits = torch.full((len(group), kmax), -1e4)
        targets = torch.zeros((len(group), kmax))
        for i, (values, tgt) in enumerate(group):
            logits[i, : len(values)] = values
            targets[i, : len(tgt)] = torch.as_tensor(tgt, dtype=torch.float32)
        log_t = torch.zeros(1, requires_grad=True)
        opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

        def closure():
            opt.zero_grad()
            loss = -(targets * torch.log_softmax(logits / log_t.exp(), -1)).sum(-1).mean()
            loss.backward()
            return loss

        opt.step(closure)
        return float(torch.clamp(log_t.exp(), TEMP_MIN, TEMP_MAX).item())

    temperatures = [fit_temperature(g) if g else 1.2 for g in samples]
    laya_cfg.update({
        "fine_tuned": True,
        "model_name": "laya-persian",
        "temperature": temperatures,
        "training": {
            "epochs_completed": EPOCHS,
            "train_items": len(train_items),
            "calib_items": len(items["calib"]),
            "test_items": len(items["test"]),
            "max_len": MAX_LEN,
            "mode": "head_only",
            "base": "convaiinnovations/laya [multilingual]",
            "fine_tuned_from_checkpoint": True,
            "data": "system_one JSONL (GPT-2-distilled soft targets)",
        },
    })
    laya_cfg.pop("temperature_by_options", None)
    out_dir = paths.LAYA_OUTPUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    save_file(
        {n: v.detach().half().cpu().contiguous() for n, v in model.state_dict().items()},
        str(out_dir / "model.safetensors"),
    )
    model.encoder.config.save_pretrained(out_dir / "encoder")
    tokenizer_l.save_pretrained(out_dir / "tokenizer")
    with open(out_dir / "rl_agent_config.json", "w") as f:
        json.dump(laya_cfg, f, indent=2)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump({"temperatures": temperatures}, f, indent=2)
    del model
    gc.collect()
    print("laya_persian saved; temperatures:", [round(t, 4) for t in temperatures])
else:
    print("loading deployed models/laya_persian")

probs, bundle_laya = serving_probs("laya", eval_idx)
results["laya"] = metrics_from_probs(probs, y_eval)
print(results["laya"])

for text in SAMPLES:
    cleaned = normalize_for_pretrained(text)
    style = compute_style_features(text, cleaned, vectorizer)
    answers = typed_answers(bundle_laya, predictors.predict_laya(bundle_laya, text, cleaned, style))
    print(f"choice={answers.choice.answer} score={answers.score.value:.2f} review={answers.noul.answer}")
del probs, bundle_laya
gc.collect()


## Summary


In [ ]:
print(pd.DataFrame(results).T.to_string())
